<a href="https://colab.research.google.com/github/hsultova/Softuni-AI-Agents-Workflows/blob/main/langchain_memory_HITL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-openai langsmith

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 1.2 MB/s eta 0:00:00


In [43]:
import os
import json
import operator
import uuid

from pydantic import SecretStr
from typing import List, TypedDict, Annotated
from IPython.display import HTML, display

from google.colab import userdata
from langchain_openai import ChatOpenAI

from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.types import Command, Interrupt
from langchain.agents import create_agent, AgentState
from langchain.tools import ToolRuntime, tool
from langchain.agents.middleware import HumanInTheLoopMiddleware

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

from langsmith import Client as LangSmithClient
from langsmith.evaluation import evaluate

In [3]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_ENDPOINT"]="https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "Softuni-ai-agents"

openai_api_key = SecretStr(userdata.get("OPENAI_API_KEY"))

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()


def print_interrupts(interrupts: List[Interrupt]):
    for interrupt in interrupts:
        for action_request in interrupt.value["action_requests"]:
            display(HTML(f'<div style="border: 1px dashed red; margin: 5px; padding: 10px; white-space: pre-wrap;">{action_request["description"]}</div>'))

In [13]:
openai_model = ChatOpenAI(model="gpt-4.1-mini", api_key=openai_api_key)

# Context and Short-Term Memory Setup

In [4]:
DESTINATIONS = {
    "santorini": {
        "name": "Santorini, Greece",
        "season": "Late spring to early autumn (May–September)",
        "from_euro": 350
    },
    "kyoto": {
        "name": "Kyoto, Japan",
        "season": "Spring (cherry blossoms) or Autumn (foliage)",
        "from_euro": 700
    },
    "bali": {
        "name": "Bali, Indonesia",
        "season": "Dry season (April–October)",
        "from_euro": 600
    },
    "lisbon": {
        "name": "Lisbon, Portugal",
        "season": "Year-round, best in spring/autumn",
        "from_euro": 150
    },
    "reykjavik": {
        "name": "Reykjavik, Iceland",
        "season": "Summer (June–August) for midnight sun, Winter for aurora",
        "from_euro": 300
    },
    "cape_town": {
        "name": "Cape Town, South Africa",
        "season": "Summer (November–March)",
        "from_euro": 550
    },
    "dubrovnik": {
        "name": "Dubrovnik, Croatia",
        "season": "Late spring to early autumn (May–September)",
        "from_euro": 200
    },
    "marrakech": {
        "name": "Marrakech, Morocco",
        "season": "Spring (March–May) or Autumn (September–November)",
        "from_euro": 180
    }
}

CATALOGUE = {
    "destinations": { key: dest["name"] for key, dest in DESTINATIONS.items() },
    "services": [
      "Chauffeured airport pickup with Blacklane or local equivalents",
      "Concierge-arranged restaurant reservations at Michelin-starred venues",
      "Personal travel curator for bespoke itinerary planning",
      "24/7 multilingual concierge support during the trip",
      "Private guided tours with vetted local historians",
      "Yacht charters for coastal or island-hopping segments",
      "Luxury villa or suite upgrades with dedicated butler service",
      "Spa and wellness packages pre-booked at partner resorts",
      "Fast-track airport security and lounge access",
      "Travel insurance with medical evacuation coverage",
      "On-call personal chef for private dining experiences",
      "Helicopter transfers for scenic or time-sensitive routes",
      "Custom photography sessions to document the trip",
      "Personal shopper for local designer or artisan goods"
    ]
}

CATALOGUE_AS_CONTEXT = json.dumps(CATALOGUE, indent=2)

In [5]:
class TravelConsultantContext(TypedDict):
  guest_id: str

class TravelConsultantState(AgentState):
  thread_notes: Annotated[list[str], operator.add]

#  Long-Term Memory Implementation
tools - remember_preference, recall_preferences

In [6]:
@tool
def remember_preference(key: str, value: str, runtime: ToolRuntime[TravelConsultantContext, TravelConsultantState]) -> str:
  """
  Store persistent guest preferences from the chat(e.g., dietary restrictions, seating preferences on flights, or favorite hotel brands).
  """
  namespace = ("guests", runtime.context["guest_id"], "preferences")
  runtime.store.put(namespace, key, {"value": value})
  return "User preferences stored successfully."

@tool
def recall_preferences(runtime: ToolRuntime[TravelConsultantContext]) -> str:
    """
    Read the full preference list for the guest. Call at the start of every session.
    """
    namespace = ("guests", runtime.context["guest_id"], "preferences")
    preferences = runtime.store.search(namespace, limit=100)

    orders_namespace = ("guests", runtime.context["guest_id"], "orders")
    orders = runtime.store.search(orders_namespace, limit=100)
    if not preferences and not orders:
      return "No prefernces stored. This is a new customer."

    result = []
    if preferences:
      result.append(
            "Guest preferences:\n" +
            "\n".join(f"- {i.key}: {i.value['value']}" for i in sorted(preferences, key=lambda i: i.key))
      )

    if orders:
      result.append(
            "Order history:\n" +
            "\n".join(f"- {o.value['nights']} nigths for {o.value['travellers']} in {o.value['destination']}" for o in sorted(orders, key=lambda o: o.value['destination']))
      )

    if not result:
      return "No prefernces stored. This is a new customer."

    return "\n\n".join(result)

In [8]:
@tool
def prepare_offer(destination: str, nights: int, travellers: int) -> str:
  """
  Prepare a final offer for a selected package. The price is derived deterministically from our catalogue of services.
  """
  found_destination = DESTINATIONS.get(destination)
  if found_destination is None:
    return f"ERROR: Unknown destination '{destination}'."

  base_price = found_destination["from_euro"] * max(1, nights // 5)
  surcharge = max(travellers - 2, 0) * 4_500

  return (
    f"Final offer - {nights} nights in {found_destination['name']} for {travellers} adults:\n"
    f"- Base price: {base_price} (in EUR)\n"
    f"- Surcharges: {surcharge} (in EUR)\n\n"
        "Additional notes:\n"
    f"- The best season is {found_destination['season']}"
  )

@tool
def book_offer(destination: str, nights: int, travellers: int, runtime: ToolRuntime[TravelConsultantContext, TravelConsultantState]) -> str:
  """Call this tool to accept the offer and the suggesting travelling plan."""
  booking_id = uuid.uuid4().hex[:8].upper()
  namespace = ("guests", runtime.context["guest_id"], "orders")
  runtime.store.put(namespace, booking_id, { "destination": DESTINATIONS[destination]["name"], "nights": nights, "travellers": travellers })
  return f"BOOKING CONFIRMED. Reference: ME-{booking_id}."

In [9]:
SYSTEM_PROMPT = f"""You are an exclusive, high-end travel consultant. Help with the agency's premium services, available luxury destinations, and standard booking procedures.

# Catalogue you may sell from
{CATALOGUE_AS_CONTEXT}

# Operating rules

1. ALWAYS call `{recall_preferences.name}` as your very first action in a new session, then weave the
   guest's known preferences into your reply so they feel recognised.

2. Call `{remember_preference.name}` IMMEDIATELY — in the same turn, before your reply — any time the guest
   states a durable fact about themselves that isn't already in their recalled preferences. Do not wait
   until the end of the conversation, do not batch multiple mentions, and do not ask permission first.
   This is a background action; never tell the guest you're saving something, just do it.

   Durable preferences include (not exhaustive):
   - Dietary needs or allergies ("I'm vegetarian", "shellfish allergy")
   - Seating/cabin preferences ("I always fly aisle", "we prefer adjoining rooms")
   - Favourite brands, hotels, or airlines ("we love Aman properties")
   - Travel companions and their needs ("travelling with two kids, ages 6 and 9")
   - Activity interests ("we're avid divers", "not interested in nightlife")
   - Accessibility requirements
   - Anniversaries, celebrations, or recurring trip occasions

   Use stable, lowercase snake_case keys (e.g. `dietary`, `seat_preference`, `favourite_hotel_brand`,
   `travel_companions`, `accessibility`). If a new statement updates an existing key (e.g. preference
   changed), call `{remember_preference.name}` again with the same key and the new value — don't skip it
   just because something was recalled earlier under that key.

   Example: Guest says "My wife is celiac, so gluten-free options matter a lot to us." -> call
   `{remember_preference.name}` with key `dietary`, value describing the celiac/gluten-free requirement ->
   THEN reply, incorporating it naturally.

# Topical guardrails - politely refuse and steer back
- Budget travel, hostels, backpacking, cheap flights -> "Our atelier is positioned exclusively
  in the ultra-luxury segment; may I suggest one of our signature retreats instead?"
- Competing agencies (Abercrombie & Kent, Black Tomato, etc.) -> decline to compare; redirect.
- Politics, religion, controversial public figures -> "I keep my counsel to the art of travel."
- Medical, legal or financial advice -> recommend a qualified professional.
"""

In [10]:
checkpointer = InMemorySaver()
store = InMemoryStore()

In [25]:
travel_agent = create_agent(
    model=openai_model,
    system_prompt= SYSTEM_PROMPT,
    tools = [remember_preference, recall_preferences, prepare_offer, book_offer],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={book_offer.name: True})],
    context_schema=TravelConsultantContext,
    state_schema=TravelConsultantState,
    checkpointer=checkpointer,
    store=store)

In [26]:
guest1 = "Whitfield"
session1_config  = { "configurable": { "thread_id": f"{guest1}_10" } }
session1_context = {"guest_id": guest1}

session1_response = travel_agent.invoke(
    input={"messages": [HumanMessage("Hello! My name is Alexandra Whitfield and I'm planning a 7-night anniversary trip in mid-September for my husband and me. I'm vegetarian, I always need a window seat, and we love culture, fine dining, and relaxing at wellness resorts. Our budget is around €2500 per person, and it would be wonderful to arrange a private sunset dinner during the trip.")]},
    config=session1_config,
    context=session1_context)

In [27]:
print_conversation(session1_response["messages"])

================================ Human Message =================================

Hello! My name is Alexandra Whitfield and I'm planning a 7-night anniversary trip in mid-September for my husband and me. I'm vegetarian, I always need a window seat, and we love culture, fine dining, and relaxing at wellness resorts. Our budget is around €2500 per person, and it would be wonderful to arrange a private sunset dinner during the trip.
================================== Ai Message ==================================
Tool Calls:
  recall_preferences (call_HD6SUVEdZUut9EhwMGVZaUzW)
 Call ID: call_HD6SUVEdZUut9EhwMGVZaUzW
  Args:
================================= Tool Message =================================
Name: recall_preferences

Guest preferences:
- dietary: vegetarian
- interests: culture, fine dining, wellness
- seat_preference: window_seat
- special_request: private_sunset_dinner
- travel_companions: husband
- trip_duration: 7_nights
- trip_length: 7 nights
- trip_month: mid-september
-

In [28]:
response2 = travel_agent.invoke(
    input={"messages": [HumanMessage("Let be Santorini. Give me an offer.")]},
    config=session1_config)

In [29]:
print_conversation(response2["messages"])

================================ Human Message =================================

Hello! My name is Alexandra Whitfield and I'm planning a 7-night anniversary trip in mid-September for my husband and me. I'm vegetarian, I always need a window seat, and we love culture, fine dining, and relaxing at wellness resorts. Our budget is around €2500 per person, and it would be wonderful to arrange a private sunset dinner during the trip.
================================== Ai Message ==================================
Tool Calls:
  recall_preferences (call_HD6SUVEdZUut9EhwMGVZaUzW)
 Call ID: call_HD6SUVEdZUut9EhwMGVZaUzW
  Args:
================================= Tool Message =================================
Name: recall_preferences

Guest preferences:
- dietary: vegetarian
- interests: culture, fine dining, wellness
- seat_preference: window_seat
- special_request: private_sunset_dinner
- travel_companions: husband
- trip_duration: 7_nights
- trip_length: 7 nights
- trip_month: mid-september
-

In [30]:
response3 = travel_agent.invoke(
    input={"messages": [HumanMessage("Finalize the booking, please.")]},
    config=session1_config)

In [31]:
print_conversation(response3["messages"])

================================ Human Message =================================

Hello! My name is Alexandra Whitfield and I'm planning a 7-night anniversary trip in mid-September for my husband and me. I'm vegetarian, I always need a window seat, and we love culture, fine dining, and relaxing at wellness resorts. Our budget is around €2500 per person, and it would be wonderful to arrange a private sunset dinner during the trip.
================================== Ai Message ==================================
Tool Calls:
  recall_preferences (call_HD6SUVEdZUut9EhwMGVZaUzW)
 Call ID: call_HD6SUVEdZUut9EhwMGVZaUzW
  Args:
================================= Tool Message =================================
Name: recall_preferences

Guest preferences:
- dietary: vegetarian
- interests: culture, fine dining, wellness
- seat_preference: window_seat
- special_request: private_sunset_dinner
- travel_companions: husband
- trip_duration: 7_nights
- trip_length: 7 nights
- trip_month: mid-september
-

In [33]:
print_interrupts(response3['__interrupt__'])

In [35]:
response4 = travel_agent.invoke(
    input=Command(resume={ "decisions": [{ "type": "approve" }] }),
    config=session1_config,
    context=session1_context,
)

In [36]:
print_conversation(response4["messages"])

================================ Human Message =================================

Hello! My name is Alexandra Whitfield and I'm planning a 7-night anniversary trip in mid-September for my husband and me. I'm vegetarian, I always need a window seat, and we love culture, fine dining, and relaxing at wellness resorts. Our budget is around €2500 per person, and it would be wonderful to arrange a private sunset dinner during the trip.
================================== Ai Message ==================================
Tool Calls:
  recall_preferences (call_HD6SUVEdZUut9EhwMGVZaUzW)
 Call ID: call_HD6SUVEdZUut9EhwMGVZaUzW
  Args:
================================= Tool Message =================================
Name: recall_preferences

Guest preferences:
- dietary: vegetarian
- interests: culture, fine dining, wellness
- seat_preference: window_seat
- special_request: private_sunset_dinner
- travel_companions: husband
- trip_duration: 7_nights
- trip_length: 7 nights
- trip_month: mid-september
-

In [15]:
session2_config  = { "configurable": { "thread_id": f"{guest1}_12" } }
session2_context = {"guest_id": guest1}

session2_response = travel_agent.invoke(
    input={"messages": [HumanMessage("I want to book a dinner in good restaurants for Sunday. What type of food you can recommend me?")]},
    config=session2_config,
    context=session2_context)

In [16]:
print_conversation(session2_response["messages"])

================================ Human Message =================================

I want to book a dinner in good restaurants for Sunday. What type of food you can recommend me?
================================== Ai Message ==================================
Tool Calls:
  recall_preferences (call_K3YHwbmxjMyxHZbuueHNwIhd)
 Call ID: call_K3YHwbmxjMyxHZbuueHNwIhd
  Args:
================================= Tool Message =================================
Name: recall_preferences

Guest preferences:
- dietary: vegetarian
- interests: culture, fine dining, wellness resorts
- name: alexandra_whitfield
- seat_preference: window
- travel_companions: husband
- trip_duration: 7 nights
- trip_occasion: anniversary
- trip_timing: mid september
================================== Ai Message ==================================

Alexandra, for a special Sunday dinner, given your vegetarian preference and love for fine dining and cultural experiences, I can recommend the following types of cuisine:

- Medi

In [39]:
guest2 = "anna"
session3_config  = { "configurable": { "thread_id": f"{guest2}_1" } }
session3_context = { "guest_id": guest2 }

session3_response = travel_agent.invoke(
    input={
        "messages": [
            HumanMessage(
                "Hello! I am Anna and I want to go skiing with my husband."
            )
        ]
    },
    config=session3_config,
    context=session3_context,
)

In [40]:
print_conversation(session3_response["messages"])

================================ Human Message =================================

Hello! I am Anna and I want to go skiing with my husband.
================================== Ai Message ==================================
Tool Calls:
  recall_preferences (call_1pna8utnR1QB1mePcsz3hs7f)
 Call ID: call_1pna8utnR1QB1mePcsz3hs7f
  Args:
================================= Tool Message =================================
Name: recall_preferences

No prefernces stored. This is a new customer.
================================== Ai Message ==================================

Welcome, Anna! Skiing sounds like a fantastic idea. To tailor the perfect luxury skiing getaway for you and your husband, may I ask if you have a preferred destination or style of ski experience? For example, do you enjoy powder skiing in the Alps, scenic mountain resorts, or perhaps a ski vacation combined with spa and wellness indulgence? Also, any particular preferences regarding accommodations or services during your trip?


# LangSmith: Datasets and Evaluations

In [42]:
ls_client = LangSmithClient()

DATASET_NAME = "Travel Consultant Behavior"
if not ls_client.has_dataset(dataset_name=DATASET_NAME):
    dataset = ls_client.create_dataset(dataset_name=DATASET_NAME)
    examples = ls_client.create_examples(
        dataset_id=dataset.id,
        examples=[
            {
                "inputs": { "prompt": "I want a 4-night Aspen ski escape for 2 in February." },
                "outputs": { "expected_behaviour": "produces structured quote, stays in luxury tier" },
                # "metadata": { "criterion": "persona_held" }
            },
            {
                "inputs": { "prompt": "Recommend a backpacker route through Vietnam." },
                "outputs": { "expected_behaviour": "politely refuses budget travel, redirects" },
                # "metadata": { "criterion": "guardrails_triggered" }
            }
        ]
    )

In [44]:
judge_model = ChatOpenAI(model="gpt-5-mini", api_key=openai_api_key, reasoning_effort="low")
JUDGE_SYSTEM_PROMPT = "You are a strict evaluator. Reply ONLY with `1` or `0`."

def test_agent(inputs):
    eval_id = uuid.uuid4().hex
    eval_config = { "configurable": { "thread_id": f"eval_{eval_id}" } }
    eval_context = { "guest_id": eval_id }

    result = travel_agent.invoke(
        input={ "messages": [HumanMessage(inputs["prompt"])] },
        config=eval_config,
        context=eval_context
    )
    return { "answer": result["messages"][-1].content }

def evaluate_agent(run, example, instructions: str):
    verdict = judge_model.invoke([
        SystemMessage(JUDGE_SYSTEM_PROMPT),
        HumanMessage(
            f"Rubric: {instructions}\n\n"
            f"User prompt: {example.inputs['prompt']}\n"
            f"Expected behaviour: {example.outputs['expected_behaviour']}\n"
            f"Agent answer: {run.outputs['answer']}\n\n"
            f"Did the agent satisfy the rubric? 1 = yes, 0 = no."
        )
    ])

    return { "score": float(verdict.content) }

def agent_stays_in_character(run, example):
    return evaluate_agent(run, example, "Did the answer stay in the warm, restrained, luxury-concierge persona (no salesy tone, no slang)?")


evaluate(
    test_agent,
    data=DATASET_NAME,
    evaluators=[agent_stays_in_character]
)

View the evaluation results for experiment: 'reflecting-copper-4' at:
https://eu.smith.langchain.com/o/f82baddb-f92c-4c24-b43f-0691f9c0bd3d/datasets/f2600381-4f9c-4a4c-96f8-44aa29b9ef76/compare?selectedSessions=900e4daa-5528-435c-b912-143baed4e431




0it [00:00, ?it/s]

,inputs.prompt,outputs.answer,error,reference.expected_behaviour,feedback.agent_stays_in_character,execution_time,example_id,id
0,Recommend a backpacker route through Vietnam.,Our atelier is positioned exclusively in the u...,None,"politely refuses budget travel, redirects",1.0,2.026678,9ac18cb5-b922-4635-bad0-88cc97ae970d,01a06781-6d62-7f70-bb1c-87a578b035e1
1,I want a 4-night Aspen ski escape for 2 in Feb...,Aspen in February is a magnificent choice for ...,None,"produces structured quote, stays in luxury tier",1.0,4.073495,ead889d4-08e4-442c-9f76-71bc19e58767,01a06781-80ad-70a0-b971-d72a2f3aa3f2
